# MarketPulse LK – Regression and comparative inference
Companion to Notebook 03. Research questions: how are returns associated across firms, and can lagged trading activity and volatility explain current absolute returns? Classical ANOVA is descriptive of OLS fit; inference uses HAC covariance because financial observations can be heteroskedastic and serially dependent. All returns are percentage log returns on matching adjacent dates in the observed union, not a verified exchange calendar.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
root = Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
out = root/'outputs/tables'
figs = root/'outputs/figures/regression'
figs.mkdir(parents=True, exist_ok=True)
assets = ['JKH','HAYL','VONE']
frames = {a:pd.read_csv(root/f'data/processed/{a}_cleaned.csv',parse_dates=['Date']).set_index('Date').sort_index() for a in assets}
calendar = pd.DatetimeIndex(sorted(set().union(*(set(d.index) for d in frames.values()))))
prices = pd.DataFrame({a:frames[a].Adjusted_Close.reindex(calendar) for a in assets})
returns = np.log(prices).diff()*100
volumes = pd.DataFrame({a:frames[a].Volume.reindex(calendar) for a in assets})
print('Observed dates:',len(calendar)); display(returns.count())


Observed dates: 3005


JKH     2944
HAYL    2473
VONE    2817
dtype: int64

## Regression hypotheses and assumptions
For each slope, H0: beta = 0; H1: beta != 0. For the overall model H0: all slopes equal zero; H1: at least one differs. Use alpha = 0.05. The same-day cross-equity models describe association, not causation or forecasting. Lagged regressors are constructed before missing rows are removed. HAC uses five lags as a prespecified working choice; missing observations make these lags approximate observed-session lags.

In [2]:
coef=[]; anova=[]; diagnostics=[]; validation=[]; vif=[]
def fit_model(name,y,X,predictive=False):
    data=pd.concat([y.rename('target'),X],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    Y=data.target; Z=sm.add_constant(data.drop(columns='target'),has_constant='add')
    model=sm.OLS(Y,Z).fit(); robust=model.get_robustcov_results(cov_type='HAC',maxlags=5,use_t=True)
    ci=robust.conf_int(); k=int(model.df_model); df=int(model.df_resid)
    for i,col in enumerate(Z.columns):
        coef.append({'Model':name,'Term':col,'Coefficient':robust.params[i],'HAC SE':robust.bse[i],'p value':robust.pvalues[i],'95% CI low':ci[i,0],'95% CI high':ci[i,1]})
    for source,ss,dof in [('Regression',model.ess,k),('Residual',model.ssr,df),('Total',model.centered_tss,k+df)]:
        anova.append({'Model':name,'Source':source,'SS':ss,'df':dof,'MS':ss/dof,'Classical F':model.fvalue if source=='Regression' else np.nan,'Classical p':model.f_pvalue if source=='Regression' else np.nan})
    bp=het_breuschpagan(model.resid,Z); jb=stats.jarque_bera(model.resid)
    diagnostics.append({'Model':name,'N':len(Y),'R2':model.rsquared,'Adjusted R2':model.rsquared_adj,'HAC joint F':float(robust.fvalue),'HAC joint p':float(robust.f_pvalue),'BP p':bp[1],'JB p':jb.pvalue,'Durbin Watson':durbin_watson(model.resid),'Ljung Box p (10)':acorr_ljungbox(model.resid,lags=[10],return_df=True).lb_pvalue.iloc[0],'Cook D max':model.get_influence().cooks_distance[0].max()})
    for i,col in enumerate(Z.columns):
        if col!='const':vif.append({'Model':name,'Term':col,'VIF':variance_inflation_factor(Z.values,i)})
    fig,ax=plt.subplots(1,2,figsize=(10,4)); ax[0].scatter(model.fittedvalues,model.resid,s=5,alpha=.35);ax[0].axhline(0,color='black');ax[0].set(xlabel='Fitted',ylabel='Residual',title=name);sm.qqplot(model.resid,line='s',ax=ax[1]);fig.tight_layout();fig.savefig(figs/f'{name}.png',dpi=150);plt.close(fig)
    if predictive:
        cut=int(.8*len(Y)); train=sm.OLS(Y.iloc[:cut],Z.iloc[:cut]).fit(); pred=train.predict(Z.iloc[cut:]); actual=Y.iloc[cut:]; baseline=pd.Series(Y.iloc[:cut].mean(),index=actual.index)
        for label,p in [('OLS',pred),('Training mean',baseline)]:
            validation.append({'Model':name,'Method':label,'Train end':Y.index[cut-1],'Test start':actual.index[0],'Test N':len(actual),'MAE':np.mean(abs(actual-p)),'RMSE':np.sqrt(np.mean((actual-p)**2))})
for a in ['HAYL','VONE']:fit_model(f'{a}_on_JKH',returns[a],returns[['JKH']])
for a in assets:
    y=returns[a].abs()
    fit_model(f'{a}_volume',y,np.log1p(volumes[[a]]).rename(columns={a:'log_volume'}))
    X=pd.DataFrame({'lag_abs_return':y.shift(1),'lag_log_volume':np.log1p(volumes[a]).shift(1),'lag_rolling_std':returns[a].rolling(20,min_periods=20).std().shift(1)})
    if a!='JKH':X['lag_JKH_return']=returns.JKH.shift(1)
    fit_model(f'{a}_multiple',y,X,True)
for name,rows in [('regression_coefficients',coef),('regression_anova',anova),('regression_diagnostics',diagnostics),('regression_vif',vif),('regression_validation',validation)]:
    table=pd.DataFrame(rows);table.to_csv(out/f'{name}.csv',index=False);display(table)


,Model,Term,Coefficient,HAC SE,p value,95% CI low,95% CI high
0,HAYL_on_JKH,const,0.098650,0.051382,5.498281e-02,-0.002107,0.199406
1,HAYL_on_JKH,JKH,0.532113,0.082415,1.287973e-10,0.370503,0.693724
2,VONE_on_JKH,const,0.059773,0.049735,2.295352e-01,-0.037749,0.157295
3,VONE_on_JKH,JKH,0.686702,0.083712,3.533246e-16,0.522558,0.850846
4,JKH_volume,const,-1.366076,0.329157,3.415698e-05,-2.011477,-0.720674
5,JKH_volume,log_volume,0.153658,0.022531,1.102180e-11,0.109480,0.197837
6,JKH_multiple,const,-0.276006,0.252017,2.735424e-01,-0.770192,0.218181
7,JKH_multiple,lag_abs_return,0.241030,0.067640,3.729400e-04,0.108394,0.373666
8,JKH_multiple,lag_log_volume,0.040750,0.017786,2.203540e-02,0.005874,0.075627
9,JKH_multiple,lag_rolling_std,0.297494,0.055935,1.139356e-07,0.187810,0.407178


,Model,Source,SS,df,MS,Classical F,Classical p
0,HAYL_on_JKH,Regression,1725.926447,1,1725.926447,293.235021,3.595196e-62
1,HAYL_on_JKH,Residual,14384.926529,2444,5.885813,NaN,NaN
2,HAYL_on_JKH,Total,16110.852976,2445,6.589306,NaN,NaN
3,VONE_on_JKH,Regression,3093.698455,1,3093.698455,409.624037,4.277482e-85
4,VONE_on_JKH,Residual,21064.010423,2789,7.552532,NaN,NaN
5,VONE_on_JKH,Total,24157.708877,2790,8.658677,NaN,NaN
6,JKH_volume,Regression,129.177999,1,129.177999,93.839393,7.232405e-22
7,JKH_volume,Residual,4049.916125,2942,1.376586,NaN,NaN
8,JKH_volume,Total,4179.094124,2943,1.420012,NaN,NaN
9,JKH_multiple,Regression,561.021298,3,187.007099,141.952008,7.099880e-85


,Model,N,R2,Adjusted R2,HAC joint F,HAC joint p,BP p,JB p,Durbin Watson,Ljung Box p (10),Cook D max
0,HAYL_on_JKH,2446,0.107128,0.106763,41.686675,1.287973e-10,7.048222e-02,0.0,1.897698,5.410774e-06,2.238372
1,VONE_on_JKH,2791,0.128063,0.127750,67.291624,3.533246e-16,1.941158e-02,0.0,1.955007,9.693558e-15,1.688304
2,JKH_volume,2944,0.030911,0.030581,46.509489,1.102180e-11,1.164728e-01,0.0,1.424836,2.426794e-198,0.101634
3,JKH_multiple,2481,0.146702,0.145669,28.546283,3.837201e-18,4.657938e-50,0.0,2.028573,1.537412e-14,2.110950
4,HAYL_volume,2473,0.081517,0.081145,63.533115,2.385960e-15,1.070073e-06,0.0,1.348376,3.185921e-169,0.239013
5,HAYL_multiple,1483,0.170677,0.168433,28.147543,1.460956e-22,1.253832e-10,0.0,2.028809,5.850687e-02,1.287647
6,VONE_volume,2817,0.117289,0.116975,91.306739,2.580533e-21,1.159054e-12,0.0,1.361759,0.000000e+00,0.110840
7,VONE_multiple,2081,0.243234,0.241776,41.040479,3.791611e-33,5.252960e-20,0.0,2.043139,1.234841e-07,0.235206


,Model,Term,VIF
0,HAYL_on_JKH,JKH,1.000000
1,VONE_on_JKH,JKH,1.000000
2,JKH_volume,log_volume,1.000000
3,JKH_multiple,lag_abs_return,1.233465
4,JKH_multiple,lag_log_volume,1.039497
5,JKH_multiple,lag_rolling_std,1.212982
6,HAYL_volume,log_volume,1.000000
7,HAYL_multiple,lag_abs_return,1.343441
8,HAYL_multiple,lag_log_volume,1.235600
9,HAYL_multiple,lag_rolling_std,1.293755


,Model,Method,Train end,Test start,Test N,MAE,RMSE
0,JKH_multiple,OLS,2024-08-12,2024-08-13,497,0.665725,0.918154
1,JKH_multiple,Training mean,2024-08-12,2024-08-13,497,0.667867,0.942182
2,HAYL_multiple,OLS,2025-07-25,2025-07-28,297,0.828938,1.127777
3,HAYL_multiple,Training mean,2025-07-25,2025-07-28,297,1.338379,1.502475
4,VONE_multiple,OLS,2025-01-21,2025-01-22,417,0.968915,1.254084
5,VONE_multiple,Training mean,2025-01-21,2025-01-22,417,1.393387,1.544281


## Paired non-parametric comparison
The Wilcoxon signed-rank test compares paired absolute returns on shared dates. H0: paired differences have a distribution symmetric around zero. H1: a nonzero location shift. Standard p-values assume independent pairs and symmetry; because market observations can be dependent, treat these as exploratory. A moving-block bootstrap of the mean paired difference supplements this analysis. It estimates a different quantity, the mean difference, and is not a replacement Wilcoxon p-value.

In [3]:
from itertools import combinations
rng=np.random.default_rng(42); rows=[]
for a,b in combinations(assets,2):
    d=(returns[a].abs()-returns[b].abs()).dropna().to_numpy(); n=len(d)
    w=stats.wilcoxon(d); L=20; blocks=int(np.ceil(n/L))
    draws=np.array([d[(rng.integers(0,n-L+1,size=blocks)[:,None]+np.arange(L)).ravel()[:n]].mean() for _ in range(2000)])
    lo,hi=np.quantile(draws,[.025,.975]);rows.append({'Pair':f'{a}-{b}','N':n,'Wilcoxon W':w.statistic,'Exploratory p':w.pvalue,'Mean difference (pp)':d.mean(),'Median difference (pp)':np.median(d),'Block bootstrap mean CI low':lo,'Block bootstrap mean CI high':hi})
paired=pd.DataFrame(rows);paired['Holm p']=multipletests(paired['Exploratory p'],method='holm')[1];paired.to_csv(out/'paired_volatility_tests.csv',index=False);display(paired)


,Pair,N,Wilcoxon W,Exploratory p,Mean difference (pp),Median difference (pp),Block bootstrap mean CI low,Block bootstrap mean CI high,Holm p
0,JKH-HAYL,2446,1072791.5,7.461851e-29,-0.506344,-0.178312,-0.680493,-0.358185,1.492370e-28
1,JKH-VONE,2791,1062013.5,4.260048e-87,-0.843614,-0.423210,-1.033392,-0.679949,1.278014e-86
2,HAYL-VONE,2373,1077031.5,5.693754e-17,-0.359250,-0.206398,-0.473404,-0.246003,5.693754e-17


## Interpretation and limitations
Use HAC p-values and confidence intervals for slope inference rather than classical ANOVA p-values when classical assumptions fail. R² measures in-sample fit, not forecasting quality. Compare test MAE/RMSE with the training mean baseline. A significant association is not evidence of causation. Absolute returns are a noisy volatility proxy. The observed trading calendar and unverified corporate-action adjustments limit the strength of conclusions. Wilcoxon results are exploratory; block length sensitivity and a verified calendar would strengthen inference.